# Resultados BBOB (COCO): los 12 algoritmos

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alexanderefe/test_transfer_in_bioalgorithms/blob/main/notebooks/analisis_bbob.ipynb)

Segundo benchmark de la tesis, aparte de CEC 2022 (que tiene su propio notebook,
`analisis_resultados.ipynb`): la suite **`bbob`** de la plataforma COCO, con su
protocolo nativo (opción A de `README_BBOB.md`).

| | |
|---|---|
| Algoritmos | Propuesto, PSO, DE, IPOP-CMA-ES, MadDE, NL-SHADE-RSP + GA, JADE, CLPSO, ABC, CLONALG, opt-aiNet |
| Funciones | 24 (f1–f24), en 5 grupos |
| Dimensiones | D = 2, 3, 5, 10, 20, 40 |
| Repeticiones | 15 instancias por función (en COCO reemplazan a las semillas) |
| Presupuesto | 10⁴·D evaluaciones por corrida |
| Corridas | 25 920, datos en `resultados_bbob/` |

**Uso:** *Entorno de ejecución → Ejecutar todas*. La primera vez clona el
repositorio y regenera tablas y figuras (unos minutos). Después, los selectores
permiten recorrer todo sin volver a calcular.

Todo el análisis es **por dimensión**: los rangos no se promedian entre
dimensiones, igual que en CEC no se promedian entre configuraciones. Cómo leer
cada tabla y figura: `LEEME.md` (se muestra al final).

## 1. Entorno

In [ ]:
import os, sys

REPO_URL = "https://github.com/alexanderefe/test_transfer_in_bioalgorithms.git"
EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    if not os.path.isdir("/content/Fase_pruebas"):
        !git clone -q --depth 1 {REPO_URL} /content/Fase_pruebas
    %cd /content/Fase_pruebas
    # opfunu: analisis/ importa los umbrales de problems/cec2022_wrapper.py
    # (setuptools<82 restaura pkg_resources). COCO (cocoex) NO hace falta.
    !pip install -q "setuptools<82" opfunu
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                      # ejecutado localmente desde notebooks/

RAIZ = os.getcwd()
sys.path.insert(0, RAIZ)
print("Repositorio en", RAIZ)

## 2. Datos

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

from analisis import bbob as B
from analisis.datos import NOMBRE_ALGORITMO
from problems.bbob_meta import DIMENSIONES_BBOB, FUNCIONES_BBOB, GRUPOS_BBOB, NOMBRES_BBOB

CARPETA = "resultados_bbob"
df = B.cargar_bbob(f"{CARPETA}/corridas.parquet")     # valida: completa, toda 'ok', un presupuesto
ALGS = B.algoritmos_de(df)
DIMS = B.dims_de(df)
nombre = lambda a: NOMBRE_ALGORITMO.get(a, a)
print(f"{len(df):,} corridas · presupuesto {df.attrs['presupuesto']}·D · k = {len(ALGS)} algoritmos: "
      + ", ".join(nombre(a) for a in ALGS))
print("Hardware:", ", ".join(df["host"].unique()), "· versión:", df["version"].str.split("|").str[:2].str.join("|").unique()[0])

## 3. Generar tablas y figuras

Corre `analisis/bbob.py`: rangos, Friedman, Nemenyi, Wilcoxon, tasa de éxito,
diagnóstico del middleware, y figuras CD, mapa de rangos, boxplots y
convergencia. Si ya están generadas se omite; `REGENERAR = True` lo fuerza.

Las ECDF de COCO (`figuras/cocopp/`) **ya vienen en el repositorio**: necesitan
los 25 920 archivos crudos del observador de COCO, que no se versionan.

In [ ]:
REGENERAR = False
DIR_ANALISIS = f"{CARPETA}/analisis"
if REGENERAR or not os.path.isfile(f"{DIR_ANALISIS}/figuras/convergencia_d40.png"):
    !python analisis/bbob.py --resultados {CARPETA}
else:
    print("Análisis ya generado en", DIR_ANALISIS)

## 4. Resultados principales

### 4.1 Rango promedio por dimensión

Rango de 1 (mejor) a 12 (peor) sobre las 24 funciones, a partir del error medio
de las 15 instancias, con empate si la diferencia es < 1e-8. Algoritmos
ordenados por el promedio de sus rangos en las 6 dimensiones (solo para
ordenar: no es un ranking en sí).

In [ ]:
T = f"{DIR_ANALISIS}/tablas"
rp = pd.read_csv(f"{T}/rangos_promedio.csv", index_col=0)
rp = rp.loc[rp.mean(axis=1).sort_values().index]
rp.index = [nombre(a) for a in rp.index]
rp.columns = [f"D={c}" for c in rp.columns]
display(rp.style.format("{:.2f}").background_gradient(cmap="RdYlGn_r", axis=0, vmin=1, vmax=12))
display(Image(filename=f"{DIR_ANALISIS}/figuras/rangos_contra_dim.png"))

### 4.2 Friedman y diferencia crítica

Con k = 12 algoritmos y N = 24 funciones, dos algoritmos son significativamente
distintos (Nemenyi, α = 0.05) si sus rangos promedio difieren en más de la CD.

In [ ]:
fr = pd.read_csv(f"{T}/friedman.csv")
display(fr.style.format({"estadistico": "{:.1f}", "p_valor": "{:.2e}", "cd_nemenyi": "{:.3f}"}))

### 4.3 Wilcoxon: el propuesto contra cada algoritmo

Por (dimensión, función), emparejado por instancia (15 pares), α = 0.05.
*better* = el propuesto tiene menos error de forma significativa. Cada
dimensión cuenta sobre 24 funciones; el total, sobre 144.

In [ ]:
w = pd.read_csv(f"{T}/wilcoxon_resumen.csv")
w["par"] = [f"{nombre(a)} vs {nombre(b)}" for a, b in zip(w.algoritmo_a, w.algoritmo_b)]
tot = w.groupby("par")[["better", "equal", "worse"]].sum()
tot["neto (better − worse)"] = tot["better"] - tot["worse"]
display(Markdown("**Total en las 6 dimensiones (de 144)**"))
display(tot.sort_values("neto (better − worse)", ascending=False)
        .style.background_gradient(cmap="RdYlGn", subset=["neto (better − worse)"], vmin=-144, vmax=144))

display(Markdown("**better / worse por dimensión** (equal = 24 − better − worse)"))
bw = w.assign(bw=w.better.astype(str) + " / " + w.worse.astype(str)).pivot(index="par", columns="dim", values="bw")
bw.columns = [f"D={c}" for c in bw.columns]
display(bw.loc[tot.sort_values("neto (better − worse)", ascending=False).index])

### 4.4 Tasa de éxito

% de las 360 corridas por dimensión (24 funciones × 15 instancias) que llegan a
error < 1e-8, el objetivo final de COCO.

In [ ]:
ex = pd.read_csv(f"{T}/exito.csv", index_col=0) * 100
ex = ex.loc[ex.mean(axis=1).sort_values(ascending=False).index]
ex.index = [nombre(a) for a in ex.index]
ex.columns = [f"D={c}" for c in ex.columns]
display(ex.style.format("{:.1f} %").background_gradient(cmap="RdYlGn", axis=None, vmin=0, vmax=100))
display(Image(filename=f"{DIR_ANALISIS}/figuras/exito_contra_dim.png"))

### 4.5 Por grupo de funciones

Rango promedio dentro de cada grupo de COCO. Elegir el algoritmo para ver cómo
le va en cada tipo de problema y dimensión.

In [ ]:
import ipywidgets as widgets

gr = pd.read_csv(f"{T}/rangos_por_grupo.csv")
orden_grupos = list(GRUPOS_BBOB)

def tabla_grupo(algoritmo):
    t = (gr[gr.algoritmo == algoritmo].pivot(index="grupo", columns="dim", values="rango_promedio")
         .reindex(orden_grupos))
    t.index = [f"{g} (f{GRUPOS_BBOB[g][0]}–f{GRUPOS_BBOB[g][-1]})" for g in t.index]
    t.columns = [f"D={c}" for c in t.columns]
    display(t.style.format("{:.2f}").background_gradient(cmap="RdYlGn_r", axis=None, vmin=1, vmax=12))

widgets.interact(tabla_grupo, algoritmo=widgets.Dropdown(
    options=[(nombre(a), a) for a in ALGS], value="middleware", description="Algoritmo"));

### 4.6 Diagnóstico del middleware

Media por corrida del propuesto (360 corridas por dimensión). R² < 0 bloquea la
transferencia en esa activación.

In [ ]:
dg = pd.read_csv(f"{T}/middleware_diagnostico.csv").set_index("dim")
dg.columns = ["activaciones Fase 2", "activaciones Fase 3", "Canal A", "Canal B", "abortos W₁",
              "instancias inyectadas", "fracción 1.ª activación", "corridas sin activación",
              "R² mediana", "% R² < 0"]
display(dg.style.format("{:.2f}"))

### 4.7 Lectura de los resultados

Resumen escrito con los resultados de esta parrilla (25 920 corridas, Mac
Studio, commit `d24ac77`). Si se regenera con otros datos, revisar las cifras
contra las tablas de arriba.

- **El propuesto supera a sus dos componentes corriendo solos.** Contra PSO
  gana en 86 de 144 (función, dimensión) y pierde en 1; contra DE gana en 53 y
  pierde en 3. Es la evidencia más directa de que la transferencia agrega
  sobre ejecutar los algoritmos por separado, y se mantiene en todas las
  dimensiones: la ventaja sobre PSO crece con D (2 → 22 funciones), y sobre DE
  también (4 → 15).
- **No alcanza al estado del arte.** NL-SHADE-RSP, IPOP-CMA-ES, JADE y MadDE
  quedan por delante en casi todas las dimensiones (el propuesto pierde en 90,
  77, 74 y 70 de 144 comparaciones contra cada uno, respectivamente). CLPSO
  también le gana en D ≥ 10.
- **Su mejor posición relativa es en dimensión baja.** En D = 2 es 3.º de 12
  (rango 4.46), detrás de NL-SHADE-RSP y JADE y por delante de IPOP-CMA-ES
  (4.75). Desde D = 5 baja a rango ~6–7 (5.88 en D = 40). Por grupo, en D = 2 le va mejor en las multimodales
  (rango 3.5–3.9) y peor en las unimodales mal condicionadas (6.2).
- **Frente a las referencias bioinspiradas** queda claramente sobre CLONALG,
  opt-aiNet y GA, y en balance positivo con ABC (60 mejor / 33 peor).
- **El middleware sí actúa en BBOB**: ~2.5–3.5 activaciones de Fase 2 por
  corrida, con Canal A y Canal B en casi todas. Solo 21–44 de 360 corridas por
  dimensión no activan nunca. La barrera de Wasserstein casi no aborta
  (< 0.1 por corrida). La mediana del R² del subrogado es 0.45 en D = 2 y
  0.66–0.92 en D ≥ 3; el 2.5–6 % de las activaciones se bloquean por R² < 0.
- **En las ECDF de COCO (sección 5.4), leídas en el presupuesto real**
  (x = 4, D = 10; valores aproximados, leídos del gráfico), IPOP-CMA-ES, MadDE
  y JADE alcanzan ~70 % de los pares (función, objetivo) y NL-SHADE-RSP ~60 %,
  contra ~35 % del propuesto,
  cerca de CLPSO y ABC (~40 %) y por encima de DE, GA (~30 %) y PSO (~14 %).
  Todos quedan lejos de la referencia *best 2009*.
- **Friedman rechaza H0 en las 6 dimensiones** (p < 10⁻²⁰): hay diferencias
  significativas entre los 12. Con CD = 3.40, el propuesto no se distingue
  estadísticamente de varios del grupo medio (ver diagramas CD).

## 5. Explorar

### 5.1 Error por función

Media, mediana, desviación y cuartiles del error final de las 15 instancias
(sin umbral). En verde, la mediana más baja.

In [ ]:
sup = pd.read_csv(f"{T}/suplementario.csv")
exf = pd.read_csv(f"{T}/exito_por_funcion.csv")

def tabla_funcion(dim, funcion):
    s = sup[(sup.dim == dim) & (sup.funcion == funcion)].set_index("algoritmo")
    e = exf[(exf.dim == dim) & (exf.funcion == funcion)].set_index("algoritmo")["instancias_resueltas"]
    t = s[["media", "mediana", "desv", "minimo", "q1", "q3", "maximo"]].join(e.rename("resueltas/15"))
    t.index = [nombre(a) for a in t.index]
    t.columns = ["media", "mediana", "desv", "mín", "P25", "P75", "máx", "resueltas/15"]
    display(Markdown(f"**f{funcion} {NOMBRES_BBOB[funcion]}, D = {dim}**"))
    display(t.sort_values("mediana").style
            .format("{:.3e}", subset=["media", "mediana", "desv", "mín", "P25", "P75", "máx"])
            .highlight_min(subset=["mediana"], color="#b7e1b7"))

widgets.interact(tabla_funcion,
                 dim=widgets.Dropdown(options=DIMS, value=10, description="D"),
                 funcion=widgets.Dropdown(options=[(f"f{f} {NOMBRES_BBOB[f]}", f) for f in FUNCIONES_BBOB],
                                          description="Función", layout=widgets.Layout(width="360px")));

### 5.2 Figuras

Paneles de las 24 funciones por dimensión. Los PNG grandes se ven mejor
abriéndolos en otra pestaña (clic derecho).

In [ ]:
from pathlib import Path
F = Path(DIR_ANALISIS) / "figuras"
TIPOS = {
    "Diagrama CD de Nemenyi": lambda d: F / f"cd_d{d:02d}.png",
    "Mapa de rangos (algoritmo × función)": lambda d: F / f"mapa_rangos_d{d:02d}.png",
    "Boxplots del error final (24 funciones)": lambda d: F / f"boxplot_d{d:02d}.png",
    "Convergencia, mediana de 15 instancias (24 funciones)": lambda d: F / f"convergencia_d{d:02d}.png",
}

def ver_figura(tipo, dim):
    ruta = TIPOS[tipo](dim)
    display(Image(filename=str(ruta))) if ruta.exists() else print("No existe", ruta, "— ¿se generó (sección 3)?")

widgets.interact(ver_figura,
                 tipo=widgets.Dropdown(options=list(TIPOS), description="Figura",
                                       layout=widgets.Layout(width="460px")),
                 dim=widgets.Dropdown(options=DIMS, value=10, description="D"));

### 5.3 Boxplot y convergencia a medida

Una función y un subconjunto de algoritmos, directo desde los datos.

In [ ]:
import matplotlib.pyplot as plt
from analisis.boxplots import _cajas

UMBRAL = 1e-8
FRACS = None

def a_medida(dim, funcion, algoritmos, grafico):
    algoritmos = [a for a in ALGS if a in algoritmos]
    if not algoritmos:
        print("Elegir al menos un algoritmo."); return
    titulo = f"f{funcion} {NOMBRES_BBOB[funcion]}, D = {dim} (15 instancias)"
    if grafico == "Boxplot":
        sub = df[(df.dim == dim) & (df.funcion == funcion)]
        datos = [np.maximum(sub.loc[sub.algoritmo == a, "error"].to_numpy(float), UMBRAL) for a in algoritmos]
        fig, ax = plt.subplots(figsize=(max(5, 0.7 * len(algoritmos) + 2), 4.5))
        _cajas(ax, datos, algoritmos, log=True, fuente_etiquetas=9)
        ax.set_ylabel("error final f − f(x*) (piso 1e-8)")
    else:
        from problems.cec2022_wrapper import FRACCIONES_CHECKPOINT
        x = np.asarray(FRACCIONES_CHECKPOINT) * df.attrs["presupuesto"] * dim
        curvas = B.curvas_convergencia(df, dim, funcion)
        idx = B._indice_en_grupo(ALGS)
        fig, ax = plt.subplots(figsize=(8, 4.8))
        for a in algoritmos:
            est = B._estilo(a, idx[a])
            ax.plot(x, curvas[a], label=nombre(a), color=est["color"], linestyle=est["linestyle"],
                    linewidth=est["linewidth"], zorder=est["zorder"])
        ax.set_xscale("log"); ax.set_yscale("log")
        ax.set_xlabel("evaluaciones"); ax.set_ylabel("error best-so-far (mediana, piso 1e-8)")
        ax.legend(fontsize=8, frameon=False, bbox_to_anchor=(1.01, 1), loc="upper left")
        ax.grid(True, alpha=0.25)
    ax.set_title(titulo)
    plt.tight_layout(); plt.show()

widgets.interact(a_medida,
                 dim=widgets.Dropdown(options=DIMS, value=10, description="D"),
                 funcion=widgets.Dropdown(options=[(f"f{f} {NOMBRES_BBOB[f]}", f) for f in FUNCIONES_BBOB],
                                          description="Función", layout=widgets.Layout(width="360px")),
                 algoritmos=widgets.SelectMultiple(options=[(nombre(a), a) for a in ALGS],
                                                   value=tuple(ALGS), rows=12, description="Algoritmos"),
                 grafico=widgets.ToggleButtons(options=["Boxplot", "Convergencia"], description="Gráfico"));

### 5.4 ECDF de COCO (`cocopp`)

La figura estándar con que se publica en BBOB: fracción de pares (función,
objetivo de precisión entre 10² y 10⁻⁸) que cada algoritmo alcanza, contra el
número de evaluaciones dividido por D (escala log). Más arriba y más a la
izquierda = mejor. *best 2009* es la envolvente de los mejores algoritmos de
BBOB 2009 (referencia del archivo de COCO).

**Ojo con el eje x:** el presupuesto real es 10⁴·D, es decir **log₁₀ = 4**, y
la cruz grande de cada curva lo marca. A la derecha de la cruz, `cocopp`
dibuja **reinicios simulados** (remuestreo de corridas, la convención de
COCO), no evaluaciones hechas. Para comparar lo que realmente pasó, leer las
curvas en x = 4.

Se generaron con `cocopp` sobre los archivos crudos del observador
(`README_MAC_BBOB.md` §8) y vienen en el repositorio.

In [ ]:
from IPython.display import SVG
DIR_ECDF = F / "cocopp"
GRUPOS_ECDF = {"Todas (f1–f24)": "todas", "Separables (f1–f5)": "separables",
               "Condicionamiento moderado (f6–f9)": "cond_moderado",
               "Condicionamiento alto, unimodales (f10–f14)": "cond_alto",
               "Multimodales, estructura adecuada (f15–f19)": "multimodales_adecuada",
               "Multimodales, estructura débil (f20–f24)": "multimodales_debil"}

def ver_ecdf(dim, grupo):
    for ext in ("png", "svg"):
        ruta = DIR_ECDF / f"ecdf_d{dim:02d}_{grupo}.{ext}"
        if ruta.exists():
            display(Image(filename=str(ruta)) if ext == "png" else SVG(filename=str(ruta)))
            return
    print("No hay ECDF de cocopp en", DIR_ECDF)

widgets.interact(ver_ecdf,
                 dim=widgets.Dropdown(options=DIMS, value=10, description="D"),
                 grupo=widgets.Dropdown(options=list(GRUPOS_ECDF.items()), description="Funciones",
                                        layout=widgets.Layout(width="460px")));

## 6. Descargar el análisis completo

In [ ]:
# Comprime tablas y figuras en un .zip y lo descarga (solo en Colab).
import shutil
zip_ruta = shutil.make_archive("analisis_bbob", "zip", DIR_ANALISIS)
print("Generado", zip_ruta)
if EN_COLAB:
    from google.colab import files
    files.download(zip_ruta)

## 7. Cómo leer cada tabla y figura

In [ ]:
display(Markdown(Path(DIR_ANALISIS, "LEEME.md").read_text(encoding="utf-8")))